In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score, confusion_matrix, silhouette_score, silhouette_samples,
)
from scipy.stats import f_oneway
from scipy.optimize import linear_sum_assignment

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)

print("imports OK")

## 1. Data Preparation

Build a TID → ground truth label mapping from the `comm` column in `event.csv`.

In [ ]:
# --- Build TID → ground-truth label from comm column in event.csv ---
df_all = pd.read_csv("event.csv")

tid_to_cmd = {}
for _, row in df_all.iterrows():
    tid = int(row["tid"])
    cmd = str(row.get("comm", ""))
    if cmd.startswith("stress-ng-"):
        raw = cmd.replace("stress-ng-", "")
        if raw.startswith("pthre"):
            label = "pthread"
        elif raw.startswith("switc"):
            label = "switch"
        elif raw == "vm":
            continue  # skip vm workload
        else:
            label = raw  # cpu, hdd, timer
        tid_to_cmd[tid] = label
    elif cmd == "slow-timer":
        tid_to_cmd[tid] = "slow_timer"

print(f"Labelled TIDs found in event.csv: {len(tid_to_cmd)}")
print("Label distribution:", pd.Series(tid_to_cmd.values()).value_counts().to_dict())

In [ ]:
# --- Filter to stress-ng tasks ---
print(f"Total tasks in event.csv: {len(df_all)}")

df = df_all[df_all["tid"].isin(tid_to_cmd)].copy()
df["label"] = df["tid"].map(tid_to_cmd)

# Metadata columns (not features for training)
META_COLUMNS = {"tid", "tgid", "ppid", "comm"}

# Features to exclude from analysis (add feature names here to remove them)
EXCLUDE_FEATURES = []

# Derive feature columns dynamically from the CSV (all columns except metadata)
FEATURE_COLUMNS = [col for col in df_all.columns if col not in META_COLUMNS and col not in EXCLUDE_FEATURES]
print(f"Feature columns ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
if EXCLUDE_FEATURES:
    print(f"Excluded features: {EXCLUDE_FEATURES}")

print()
print(df["label"].value_counts())
df.head()

In [ ]:
# --- Save clean dataset ---
df.to_csv("stress_ng_features.csv", index=False)
print("Saved stress_ng_features.csv")
df[FEATURE_COLUMNS + ["label"]].describe()

## 2. K-means Classification (K=6)

Normalize features, run K-means with K=6 (matching the 6 workload categories), and evaluate against ground truth.

In [ ]:
# --- Normalize & run K-means ---
N_CLUSTERS = 6

X = df[FEATURE_COLUMNS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
kmeans.fit(X_scaled)
df["cluster"] = kmeans.labels_

# Encode ground truth as integers
label_names = sorted(df["label"].unique())
label_to_int = {l: i for i, l in enumerate(label_names)}
y_true = df["label"].map(label_to_int).values
y_pred = kmeans.labels_

# --- Metrics ---
ari = adjusted_rand_score(y_true, y_pred)
sil = silhouette_score(X_scaled, y_pred)
print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Silhouette Score:          {sil:.4f}")
print(f"Ground truth labels: {label_names}")

In [ ]:
# --- Confusion matrix (cluster vs ground truth) ---
cm = confusion_matrix(y_true, y_pred)

# Use Hungarian algorithm to find best cluster→label mapping
row_ind, col_ind = linear_sum_assignment(-cm)
cluster_to_label = {col_ind[i]: label_names[row_ind[i]] for i in range(len(row_ind))}
print("Best cluster → label mapping (Hungarian):")
for c in sorted(cluster_to_label):
    print(f"  Cluster {c} → {cluster_to_label[c]}")

# Accuracy under best mapping
mapped_pred = np.array([label_to_int[cluster_to_label[c]] for c in y_pred])
accuracy = np.mean(mapped_pred == y_true)
print(f"\nClassification accuracy (best mapping): {accuracy:.4f}")

## 3. Visualizations

### 3.1 2D Scatter Plot (t-SNE): color = K-means cluster, marker = ground truth

In [ ]:
# --- t-SNE 2D projection ---
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(df)-1))
X_2d = tsne.fit_transform(X_scaled)

markers = {"cpu": "o", "hdd": "D", "switch": "^", "timer": "v", "pthread": "P", "slow_timer": "X"}
palette = sns.color_palette("tab10", N_CLUSTERS)

fig, ax = plt.subplots(figsize=(12, 8))
for label in label_names:
    mask = df["label"].values == label
    for cluster_id in range(N_CLUSTERS):
        cmask = mask & (df["cluster"].values == cluster_id)
        if cmask.sum() == 0:
            continue
        ax.scatter(
            X_2d[cmask, 0], X_2d[cmask, 1],
            c=[palette[cluster_id]], marker=markers[label],
            s=60, alpha=0.7, edgecolors="k", linewidths=0.3,
        )

# Legends
from matplotlib.lines import Line2D
cluster_handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=palette[i],
                          markersize=10, label=f"Cluster {i}") for i in range(N_CLUSTERS)]
label_handles = [Line2D([0], [0], marker=markers[l], color="grey", linestyle="None",
                        markersize=10, label=l) for l in label_names]
leg1 = ax.legend(handles=cluster_handles, title="K-means cluster", loc="upper left")
ax.add_artist(leg1)
ax.legend(handles=label_handles, title="Ground truth", loc="upper right")
ax.set_title("t-SNE projection — color = K-means cluster, marker = ground truth")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

### 3.2 Confusion Matrix Heatmap (cluster vs ground truth)

In [ ]:
# --- Confusion matrix heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"C{i}" for i in range(N_CLUSTERS)],
            yticklabels=label_names, ax=ax)
ax.set_xlabel("K-means Cluster")
ax.set_ylabel("Ground Truth Label")
ax.set_title("Confusion Matrix: Ground Truth vs K-means Cluster")
plt.tight_layout()
plt.show()

### 3.3 Bar Chart: Composition of each cluster (% of each task type)

In [ ]:
# --- Cluster composition bar chart ---
ct = pd.crosstab(df["cluster"], df["label"], normalize="index") * 100
ct = ct[label_names]  # consistent order

ct.plot(kind="bar", stacked=True, figsize=(10, 6), colormap="Set2", edgecolor="k", linewidth=0.5)
plt.ylabel("Percentage (%)")
plt.xlabel("K-means Cluster")
plt.title("Cluster Composition by Ground Truth Task Type")
plt.legend(title="Task type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 3.4 Elbow Method + Silhouette Score

In [ ]:
# --- Elbow method + silhouette ---
K_range = range(2, 13)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(K_range), inertias, "bo-")
ax1.axvline(x=N_CLUSTERS, color="r", linestyle="--", label=f"K={N_CLUSTERS} (ground truth)")
ax1.set_xlabel("K")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Method")
ax1.legend()

ax2.plot(list(K_range), sil_scores, "go-")
ax2.axvline(x=N_CLUSTERS, color="r", linestyle="--", label=f"K={N_CLUSTERS} (ground truth)")
ax2.set_xlabel("K")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score vs K")
ax2.legend()

plt.tight_layout()
plt.show()

best_k_sil = list(K_range)[np.argmax(sil_scores)]
print(f"Best K by silhouette: {best_k_sil} (score={max(sil_scores):.4f})")
k_idx = list(K_range).index(N_CLUSTERS)
print(f"K={N_CLUSTERS} silhouette: {sil_scores[k_idx]:.4f}")

## 4. Feature Importance Analysis

### 4.1 ANOVA F-score bar chart

In [ ]:
# --- ANOVA F-score for each feature ---
groups_by_label = df.groupby("label")
f_scores = {}
p_value = {}
for feat in FEATURE_COLUMNS:
    groups = [g[feat].values for _, g in groups_by_label]
    f_stat, p_val = f_oneway(*groups)
    f_scores[feat] = f_stat
    p_value[feat] = p_val

real_f_df = pd.DataFrame({"feature": list(f_scores.keys()), "F_score": list(f_scores.values())})
real_p_value = pd.DataFrame({"feature": list(p_value.keys()), "F_score": list(p_value.values())})
f_df = pd.DataFrame({"feature": list(f_scores.keys()), "F_score": np.log10(list(f_scores.values()))})
f_df = f_df.sort_values("F_score", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(f_df["feature"], f_df["F_score"], color="steelblue", edgecolor="k")
ax.set_xlabel("ANOVA F-score")
ax.set_title("Feature Importance: ANOVA F-score (between-class separability)")
plt.tight_layout()
plt.show()

print(real_p_value)
real_f_df

### 4.2 Box Plots per Feature (grouped by ground truth label)

In [ ]:
# --- Box plots per feature ---
n_feats = len(FEATURE_COLUMNS)
ncols = 3
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLUMNS):
    sns.boxplot(data=df, x="label", y=feat, ax=axes[i], order=label_names, palette="Set2")
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions by Ground Truth Label", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Correlation Matrix Heatmap

In [ ]:
# --- Correlation matrix ---
corr = df[FEATURE_COLUMNS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### 4.5 Single-Feature Silhouette Test

In [ ]:
# --- Single-feature silhouette test ---
single_sil = {}
for feat in FEATURE_COLUMNS:
    X_single = X_scaled[:, FEATURE_COLUMNS.index(feat)].reshape(-1, 1)
    km_single = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
    km_single.fit(X_single)
    n_unique = len(set(km_single.labels_))
    if n_unique >= 2:
        single_sil[feat] = silhouette_score(X_single, km_single.labels_)
    else:
        single_sil[feat] = 0.0

sil_df = pd.DataFrame({"feature": list(single_sil.keys()), "silhouette": list(single_sil.values())})
sil_df = sil_df.sort_values("silhouette", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#e74c3c" if s < 0.2 else "#f39c12" if s < 0.4 else "#2ecc71" for s in sil_df["silhouette"]]
ax.barh(sil_df["feature"], sil_df["silhouette"], color=colors, edgecolor="k")
ax.axvline(x=0.25, color="grey", linestyle="--", alpha=0.7, label="threshold=0.25")
ax.set_xlabel(f"Silhouette Score (K={N_CLUSTERS}, single feature)")
ax.set_title("Single-Feature Silhouette Test")
ax.legend()
plt.tight_layout()
plt.show()